# MuSeg-AI Thigh Segmentation — Augmented Dataset (Lambda)

Runs [fabianbalsiger/museg-ai](https://github.com/fabianbalsiger/museg-ai) (`thigh-model3`)
on the 20 augmented NIfTI water volumes.

Input is already NIfTI — no DICOM conversion needed. Each NIfTI water file is loaded
directly with `Volume.load(nii_path)` and passed as both input channels (water-only mode).

Data: `~/our_augmented_dataset/{stem}_augmented000_water.nii.gz`
Output: `~/museg_augmented_segs/{stem}_dseg.nii.gz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/our_augmented_dataset/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/our_augmented_dataset/
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/museg_augmented_segs/ \
  /path/to/local/museg/augmented_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys, importlib, os, site

def sh(cmd, fatal=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    out = (r.stdout + r.stderr).strip()
    if r.returncode != 0:
        print(f'[{"ERROR" if fatal else "WARN"}] {out[:400]}')
        if fatal:
            raise RuntimeError(f'Command failed: {cmd}')
    else:
        print(f'OK: {cmd[:70]}')
    return r.returncode == 0

# ── Docker ────────────────────────────────────────────────────────────────────
r = subprocess.run('which docker', shell=True, capture_output=True)
if r.returncode != 0:
    sh('sudo apt-get update -qq')
    sh('sudo apt-get install -y docker.io')
else:
    print('Docker present:', r.stdout.strip())
sh('sudo systemctl start docker')
sh('sudo chmod 666 /var/run/docker.sock')

# ── Clone repo ────────────────────────────────────────────────────────────────
CLONE_DIR = '/tmp/museg_ai_src'
if not os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    sh(f'git clone https://github.com/fabianbalsiger/museg-ai.git {CLONE_DIR}', fatal=True)
else:
    sh(f'git -C {CLONE_DIR} pull')

# ── Install (regular, not editable — pyproject.toml lacks build_editable hook) ─
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', CLONE_DIR])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK'])
importlib.invalidate_caches()

import musegai
print(f'museg-ai ready  version={getattr(musegai, "__version__", "?")}') 

In [ ]:
import docker
try:
    client = docker.from_env()
    client.ping()
    print('Docker is running.')
except Exception as e:
    raise RuntimeError(f'Docker not reachable — re-run setup cell.\n{e}')

In [ ]:
import glob
import numpy as np
import SimpleITK as sitk
from musegai import api
from musegai.api import Volume

DATA_DIR   = os.path.expanduser('~/our_augmented_dataset')
OUTPUT_DIR = os.path.expanduser('~/museg_augmented_segs')

os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL_MAP = {
    1:  'Vastus_Lateralis',   2:  'Vastus_Intermedius',
    3:  'Vastus_Medialis',    4:  'Rectus_Femoris',
    5:  'Sartorius',          6:  'Gracilis',
    7:  'Semimembranosus',    8:  'Semitendinosus',
    9:  'Biceps_Femoris',     10: 'Biceps_Femoris_Short',
    11: 'Adductor_Magnus',    12: 'Adductor_Longus',
    13: 'Adductor_Brevis',
}

nii_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_augmented*_water.nii.gz')))
print(f'Found {len(nii_files)} NIfTI water volumes')

In [ ]:
for nii_path in nii_files:
    basename = os.path.basename(nii_path)
    stem     = basename.replace('_water.nii.gz', '')
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_dseg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing: {stem}')
    # Load NIfTI directly — no DICOM conversion needed
    img_vol = Volume.load(nii_path)
    print(f'  Shape: {img_vol.shape}  Spacing: {img_vol.spacing}')

    # Water-only mode: pass the same volume as both input channels
    results, labels = api.segment_volumes(
        {stem: [img_vol, img_vol]},
        model='thigh-model3',
        side='left+right',
    )

    segmentation = results[stem]
    segmentation.save(out_path)

    seg_arr = segmentation.array
    print(f'  Labels: {sorted(np.unique(seg_arr).tolist())}')
    for idx_lbl, name in LABEL_MAP.items():
        n = int((seg_arr == idx_lbl).sum())
        if n > 0:
            print(f'    {idx_lbl:<4} {name:<25} {n:>10,}')
    print(f'  Saved → {out_path}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*_dseg.nii.gz')))
print(f'Output files: {len(results)} / {len(nii_files)}')
if results:
    sample = Volume.load(results[0])
    print(f'Sample : {results[0]}')
    print(f'Shape  : {sample.shape}')
    print(f'Labels : {sorted(np.unique(sample.array).tolist())}')